<a href="https://colab.research.google.com/github/IgorForceBR0202/linguagens_programacao/blob/main/Atividade_Pratica_RH_com_SQLAlchemy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Atividade Prática: Desenvolvendo um Sistema de RH com SQLAlchemy**

Nível 1: Básico — Configuração e SQL Puro com Segurança
Nesta etapa, os alunos farão a conexão inicial e utilizarão SQL nativo, compreendendo a importância de proteger o banco de dados contra ataques.


In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd

# Passo 1: Criar a conexão com o banco SQLite
engine = create_engine("sqlite:///sistema_rh.db")


# Passo 2: Criar a tabela funcionarios usando SQL puro
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome VARCHAR(100) NOT NULL,
            cargo VARCHAR(100) NOT NULL,
            salario FLOAT NOT NULL
        )
    """))
# Passo 3: Inserir funcionário utilizando parâmetros seguros
nome = "Igor"
cargo = "Estagiário de TI"
salario = 1500.00

with engine.begin() as conn:
    conn.execute(
        text("""
            INSERT INTO funcionarios (nome, cargo, salario)
            VALUES (:nome, :cargo, :salario)
        """),
        {
            "nome": nome,
            "cargo": cargo,
            "salario": salario
        }
    )


# Passo 4: Consultar os funcionários com Pandas
df = pd.read_sql_query(
    "SELECT * FROM funcionarios",
    engine
)

print(df)

   id  nome             cargo  salario
0   1  Igor  Estagiário de TI   1500.0


### **Nível 2: Intermediário — SQLAlchemy Core (Automatização Programática)**

O objetivo agora é abandonar o SQL em texto e usar as estruturas Python do SQLAlchemy Core, ideais para scripts de manipulação de dados e relatórios.


In [ ]:
from sqlalchemy import (
    create_engine,
    MetaData,
    Table,
    Column,
    Integer,
    String,
    Float,
    insert,
    update,
    select,
    func
)

# Conexão com o banco
engine = create_engine("sqlite:///sistema_rh.db")

# Metadata
metadata = MetaData()

# PASSO 1 — Criar a tabela projetos programaticamente

projetos = Table(
    "projetos",
    metadata,
    Column("id", Integer, primary_key=True, autoincrement=True),
    Column("nome", String(100), nullable=False),
    Column("responsavel", String(100), nullable=False),
    Column("orcamento", Float, nullable=False)
)

# Criar fisicamente a tabela no banco
metadata.create_all(engine)

# PASSO 2 — Inserção em lote (Bulk Insert)

lista_de_projetos = [
    {
        "nome": "Sistema de RH",
        "responsavel": "Igor",
        "orcamento": 15000.00
    },
    {
        "nome": "Dashboard EMEDI",
        "responsavel": "Igor",
        "orcamento": 10000.00
    },
    {
        "nome": "Sistema Financeiro",
        "responsavel": "Maria",
        "orcamento": 25000.00
    }
]

with engine.begin() as conn:
    conn.execute(
        insert(projetos),
        lista_de_projetos
    )

# PASSO 3 — Reajuste salarial

# Importante: refletir a tabela funcionarios existente
funcionarios = Table(
    "funcionarios",
    metadata,
    autoload_with=engine
)

reajuste = (
    update(funcionarios)
    .where(funcionarios.c.cargo == "Desenvolvedor Júnior")
    .values(
        salario=funcionarios.c.salario * 1.10
    )
)

with engine.begin() as conn:
    conn.execute(reajuste)

# PASSO 4 — Relatório salarial por cargo

relatorio = (
    select(
        funcionarios.c.cargo,
        func.avg(funcionarios.c.salario).label("media_salarial")
    )
    .group_by(funcionarios.c.cargo)
)

with engine.connect() as conn:
    resultado = conn.execute(relatorio)

    for linha in resultado:
        print(
            f"Cargo: {linha.cargo} | "
            f"Média salarial: R$ {linha.media_salarial:.2f}"
        )

Cargo: Estagiário de TI | Média salarial: R$ 1500.00


### **Nível 3: Avançado — ORM (Orientação a Objetos e Relacionamentos)**

Na fase final, a turma aplicará o padrão ORM (Object-Relational Mapping), que é amplamente utilizado no desenvolvimento de aplicações modernas.
